In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import argparse
import os
import random
import argparse
from karateclub import NetLSD, Graph2Vec, FeatherGraph
from grakel import GraphKernel
import toper

from sklearn.manifold import MDS
from sklearn.decomposition import PCA

from magnipy.magnitude.distances import get_dist

from diversity_curves.process import get_mag_graphs, get_funs_df, get_mag_funs_graphs
from diversity_curves.simulate import simulate_graphs_kwargs
from diversity_curves.classifiers import knn_classifier
from diversity_curves.clustering_performance import compute_clustering_scores_resample, compute_silhouette_score_resamples, compute_rand_agglo, compute_rand_kmeans, all_clustering_results_euclidean
from diversity_curves.baselines.spectral_zoo import est_moment
from diversity_curves.baselines.ph import obtain_input_from_graphs

# scratch space creating script

In [11]:
if True:
    np.random.seed(43)
    random.seed(314)

    # results_dir = args.output_dir
    n_low = 10
    n_high = 30
    n_steps = 1
    n_graphs = 3
    n_reps = [1,3,5]
    n_seeds = 3

    # Setting the parameters as reported in the paper
    model_params = {
        "ER": {
            "ER_probs": [0.75],
            "ER_nps": None,
        },
        "RPG": {
            "RPG_probs": [[0.9, 0.1]],
            "RPG_props": [1 / 3, 1 / 3],
        },
        "SBM": {"SBM_probs": [0.8, 0.05], "SBM_blocks": [[1 / 4, 1 / 4, 1 / 4]]},
        "RG": {"RG_probs": [0.25]},
    }

    # choice of graph metric we use for the spread computation
    graph_metrics = ["shortest_path_distance"]

    # get configuration of all models
    models = list(model_params.keys())
    all_configs = {}
    for model in models:
        model_config = tuple(
            model_params[model][param] for param in model_params[model].keys()
        )
        all_configs[model] = model_config
    configuration = tuple(all_configs[model] for model in models)

    noderange = range(n_low, n_high, n_steps)
    downsample_to = None

    scores = [
        "acc_mean",
        "silh_mean",
        "ari_agglo_mean",
        "db_mean",
        "ch_mean",
        "ari_kmeans_mean",
    ]
    scores_asc = {
        "acc_mean": False,
        "silh_mean": False,
        "ari_agglo_mean": False,
        "db_mean": True,
        "ch_mean": False,
        "ari_kmeans_mean": False,
    }

    res_dict = {}

    # if not os.path.exists(results_dir):
    #     os.makedirs(results_dir, exist_ok=True)

    # generate the graphs of all models and create label per model
    G_nx = []
    Gs = []
    n_nodes = []
    y_models = []
    class_labels = {}
    for i_m, model in enumerate(model_params):
        model_kwargs = model_params[model]
        print(f"model {model} kwargs {model_kwargs}")
        G_nx_model, Gs_model, y_model, y_probs_model, n_nodes_model = (
            simulate_graphs_kwargs(
                model=model,
                n_graphs=n_graphs,
                nodes_range=noderange,
                **model_kwargs,
            )
        )
        G_nx.extend(G_nx_model)
        Gs.extend(Gs_model)
        n_nodes.extend(n_nodes_model)
        y_models.extend([i_m] * len(G_nx_model))
        class_labels[i_m] = model

    upsample_to = max(n_nodes)
    y = y_models
    hue_names = [class_labels[i] for i in y]

    # setting parameters for plots
    sns.set_style("white")
    matplotlib.rcParams.update({"font.size": 20})

    ##################################
    ### computing diversity curves ###
    ##################################

    for metric in graph_metrics:

        for i_seed in range(n_seeds):

            res_dict[(f"divcurves_{metric}", i_seed)] = {}
            res_dict[(f"divcurves_pca_{metric}", i_seed)] = {}

            for n_rep in n_reps:
                
                print(f"Computing MAG with metric: {metric} and nrep: {n_rep} seed: {i_seed}")

                def dist_fn(G):
                    return get_dist(G=G, X=None, X2=None, metric=metric, mode="structure")

                all_spreads, weights_lists = get_mag_graphs(
                    G_nx,
                    dist_fn,
                    ts=[1],
                    upsample_to=upsample_to,
                    downsample_to=downsample_to,
                    n_repeats=n_rep,
                    scores_method='random',
                    mode='structure',
                    set_seed=False
                )

                mag_funs = get_funs_df(all_spreads)
                mag_funs_interpolated = mag_funs.interpolate(method="linear", axis="columns")



                # compute clustering metrics based on euclidean distance of diversity curve vectors
                ari_kmeans_mean, ari_kmeans_std = compute_rand_kmeans(
                    np.array(mag_funs_interpolated), y
                )
                ari_agglo_mean, ari_agglo_std = compute_rand_agglo(
                    np.array(mag_funs_interpolated), y, metric="euclidean"
                )

                means, stds = compute_clustering_scores_resample(
                    np.array(mag_funs_interpolated), y
                )

                (
                    res_dict[(f"divcurves_{metric}", i_seed)][("silh_mean", n_rep)],
                    res_dict[(f"divcurves_{metric}", i_seed)][("db_mean", n_rep, )],
                    res_dict[(f"divcurves_{metric}", i_seed)][("ch_mean", n_rep)],
                ) = means
                (
                    res_dict[(f"divcurves_{metric}", i_seed)][("silh_std", n_rep)],
                    res_dict[(f"divcurves_{metric}", i_seed)][("db_std", n_rep)],
                    res_dict[(f"divcurves_{metric}", i_seed)][("ch_std", n_rep)],
                ) = stds
                res_dict[(f"divcurves_{metric}", i_seed)][("ari_agglo_mean", n_rep)] = ari_agglo_mean
                res_dict[(f"divcurves_{metric}", i_seed)][("ari_agglo_std", n_rep)] = ari_agglo_std
                res_dict[(f"divcurves_{metric}", i_seed)][("ari_kmeans_mean", n_rep)] = ari_kmeans_mean
                res_dict[(f"divcurves_{metric}", i_seed)][("ari_kmeans_std", n_rep)] = ari_kmeans_std

                # computing accuracies
                if n_graphs > 1:
                    test_accs = knn_classifier(
                        np.array(mag_funs_interpolated),
                        np.array(y),
                        precomputed=False,
                        groups=np.array(n_nodes),
                    )
                else:
                    test_accs = knn_classifier(
                        np.array(mag_funs_interpolated), np.array(y), precomputed=False
                    )

                res_dict[(f"divcurves_{metric}", i_seed)][("acc_mean", n_rep)] = np.mean(test_accs)
                res_dict[(f"divcurves_{metric}", i_seed)][("acc_std", n_rep)] = np.std(test_accs)

                # doing PCA with Euclidean distance between the diversity curves
                pca_embedding = PCA(n_components=2)
                X_pca = pca_embedding.fit_transform(np.array(mag_funs_interpolated))

                # computing clustering scores and accuracies on transformed data from PCA
                if n_graphs > 1:
                    test_accs = knn_classifier(
                        X_pca, np.array(y), precomputed=False, groups=np.array(n_nodes)
                    )
                else:
                    test_accs = knn_classifier(X_pca, np.array(y), precomputed=False)

                res_dict[(f"divcurves_pca_{metric}", i_seed)][("acc_mean", n_rep)] = np.mean(test_accs)
                res_dict[(f"divcurves_pca_{metric}", i_seed)][("acc_std", n_rep)] = np.std(test_accs)
                # all_clustering_results_euclidean(f"divcurves_pca_{metric}_nreps_{n_rep}", res_dict, X_pca, y)
                ari_kmeans_mean, ari_kmeans_std = compute_rand_kmeans(
                    X_pca, y
                )
                ari_agglo_mean, ari_agglo_std = compute_rand_agglo(
                    X_pca, y, metric="euclidean"
                )

                means, stds = compute_clustering_scores_resample(
                    X_pca, y
                )

                (
                    res_dict[(f"divcurves_pca_{metric}", i_seed)][("silh_mean", n_rep)],
                    res_dict[(f"divcurves_pca_{metric}", i_seed)][("db_mean", n_rep, )],
                    res_dict[(f"divcurves_pca_{metric}", i_seed)][("ch_mean", n_rep)],
                ) = means
                (
                    res_dict[(f"divcurves_pca_{metric}", i_seed)][("silh_std", n_rep)],
                    res_dict[(f"divcurves_pca_{metric}", i_seed)][("db_std", n_rep)],
                    res_dict[(f"divcurves_pca_{metric}", i_seed)][("ch_std", n_rep)],
                ) = stds
                res_dict[(f"divcurves_pca_{metric}", i_seed)][("ari_agglo_mean", n_rep)] = ari_agglo_mean
                res_dict[(f"divcurves_pca_{metric}", i_seed)][("ari_agglo_std", n_rep)] = ari_agglo_std
                res_dict[(f"divcurves_pca_{metric}", i_seed)][("ari_kmeans_mean", n_rep)] = ari_kmeans_mean
                res_dict[(f"divcurves_pca_{metric}", i_seed)][("ari_kmeans_std", n_rep)] = ari_kmeans_std
                
                # # creating plots
                # fig = sns.scatterplot(
                #     x=X_pca[:, 0], y=X_pca[:, 1], hue=hue_names, palette="tab10"
                # )
                # fig.legend(frameon=False, loc=(1.04, 0), bbox_to_anchor=(1, 0.5))
                # sns.despine()
                # plt.savefig(
                #     results_dir + f"divcurves_pca_{metric}_nreps_{n_rep}.pdf",
                #     dpi=300,
                #     format="pdf",
                #     bbox_inches="tight",
                # )
                # plt.savefig(
                #     results_dir + f"divcurves_pca_{metric}_nreps_{n_rep}.png",
                #     dpi=300,
                #     format="png",
                #     bbox_inches="tight",
                # )
                # plt.close()

        
    # creating dataframes and saving results
    res_df = pd.DataFrame.from_dict(res_dict, orient="index")

    df_n_nodes = pd.DataFrame({"n_nodes": n_nodes})

model ER kwargs {'ER_probs': [0.75], 'ER_nps': None}
model RPG kwargs {'RPG_probs': [[0.9, 0.1]], 'RPG_props': [0.3333333333333333, 0.3333333333333333]}
model SBM kwargs {'SBM_probs': [0.8, 0.05], 'SBM_blocks': [[0.25, 0.25, 0.25]]}
model RG kwargs {'RG_probs': [0.25]}
Computing MAG with metric: shortest_path_distance and nrep: 1 seed: 0
Silhouette Score: 0.422 ± 0.033
Davies-Bouldin Score: 0.754 ± 0.058
Calinski-Harabasz Score: 192.721 ± 33.388
Fold 0: test accuracy = 0.8333 (k=5)
Fold 1: test accuracy = 0.7917 (k=5)
Fold 2: test accuracy = 0.8750 (k=5)
Fold 3: test accuracy = 0.8333 (k=5)
Fold 4: test accuracy = 0.8750 (k=5)
Fold 5: test accuracy = 0.9167 (k=5)
Fold 6: test accuracy = 0.8333 (k=5)
Fold 7: test accuracy = 0.9583 (k=5)
Fold 8: test accuracy = 0.8333 (k=5)
Fold 9: test accuracy = 0.8333 (k=5)

Mean test accuracy: 0.8583
Std test accuracy:  0.0464
Fold 0: test accuracy = 0.8333 (k=5)
Fold 1: test accuracy = 0.7917 (k=5)
Fold 2: test accuracy = 0.9167 (k=5)
Fold 3: test a

In [24]:
pd.MultiIndex.from_product([scores+[score[:-4]+'std' for score in scores],n_reps])

MultiIndex([(       'acc_mean', 1),
            (       'acc_mean', 3),
            (       'acc_mean', 5),
            (      'silh_mean', 1),
            (      'silh_mean', 3),
            (      'silh_mean', 5),
            ( 'ari_agglo_mean', 1),
            ( 'ari_agglo_mean', 3),
            ( 'ari_agglo_mean', 5),
            (        'db_mean', 1),
            (        'db_mean', 3),
            (        'db_mean', 5),
            (        'ch_mean', 1),
            (        'ch_mean', 3),
            (        'ch_mean', 5),
            ('ari_kmeans_mean', 1),
            ('ari_kmeans_mean', 3),
            ('ari_kmeans_mean', 5),
            (        'acc_std', 1),
            (        'acc_std', 3),
            (        'acc_std', 5),
            (       'silh_std', 1),
            (       'silh_std', 3),
            (       'silh_std', 5),
            (  'ari_agglo_std', 1),
            (  'ari_agglo_std', 3),
            (  'ari_agglo_std', 5),
            (         'db_st

In [25]:
res_df.columns = pd.MultiIndex.from_product([scores+[score[:-4]+'std' for score in scores],n_reps])

In [26]:
res_df

acc_mean                        \
                                               1         3           5   
divcurves_shortest_path_distance     0  0.421604  0.754183  192.720745   
divcurves_pca_shortest_path_distance 0  0.434270  0.732491  197.008837   
divcurves_shortest_path_distance     1  0.418545  0.762476  190.735427   
divcurves_pca_shortest_path_distance 1  0.430249  0.742956  194.602720   
divcurves_shortest_path_distance     2  0.418545  0.762476  190.735427   
divcurves_pca_shortest_path_distance 2  0.430249  0.742956  194.602720   

                                       silh_mean                       \
                                               1         3          5   
divcurves_shortest_path_distance     0  0.033456  0.058361  33.387778   
divcurves_pca_shortest_path_distance 0  0.034245  0.058968  34.709484   
divcurves_shortest_path_distance     1  0.033549  0.059957  33.126113   
divcurves_pca_shortest_path_distance 1  0.034384  0.060510  34.370930   
divcurves_shortest_path_distance     2  0.033549  0.059957  33.126113   
divcurves_pca_shortest_path_distance 2  0.034384  0.060510  34.370930   

                                       ari_agglo_mean                      \
                                                    1         3         5   
divcurves_shortest_path_distance     0       0.315380  0.115946  0.500475   
divcurves_pca_shortest_path_distance 0       0.322917  0.122553  0.502109   
divcurves_shortest_path_distance     1       0.315504  0.105647  0.498360   
divcurves_pca_shortest_path_distance 1       0.314268  0.115486  0.498357   
divcurves_shortest_path_distance     2       0.315504  0.105647  0.498360   
divcurves_pca_shortest_path_distance 2       0.314268  0.115486  0.498357   

                                         db_mean  ... ari_agglo_std    db_std  \
                                               1  ...             5         1   
divcurves_shortest_path_distance     0  0.081177  ...    195.138790  0.033980   
divcurves_pca_shortest_path_distance 0  0.078131  ...    197.968147  0.034544   
divcurves_shortest_path_distance     1  0.080853  ...    195.138790  0.033980   
divcurves_pca_shortest_path_distance 1  0.080505  ...    197.968147  0.034544   
divcurves_shortest_path_distance     2  0.080853  ...    195.138790  0.033980   
divcurves_pca_shortest_path_distance 2  0.080505  ...    197.968147  0.034544   

                                                               ch_std  \
                                               3          5         1   
divcurves_shortest_path_distance     0  0.059903  34.090992  0.313527   
divcurves_pca_shortest_path_distance 0  0.060014  34.963096  0.329558   
divcurves_shortest_path_distance     1  0.059903  34.090992  0.313527   
divcurves_pca_shortest_path_distance 1  0.060014  34.963096  0.329558   
divcurves_shortest_path_distance     2  0.059903  34.090992  0.313527   
divcurves_pca_shortest_path_distance 2  0.060014  34.963096  0.329558   

                                                           ari_kmeans_std  \
                                               3         5              1   
divcurves_shortest_path_distance     0  0.108197  0.496749       0.077670   
divcurves_pca_shortest_path_distance 0  0.121440  0.498456       0.074992   
divcurves_shortest_path_distance     1  0.108197  0.496749       0.077670   
divcurves_pca_shortest_path_distance 1  0.121440  0.498456       0.074992   
divcurves_shortest_path_distance     2  0.108197  0.496749       0.077670   
divcurves_pca_shortest_path_distance 2  0.121440  0.498456       0.074992   

                                                            
                                               3         5  
divcurves_shortest_path_distance     0  0.904167  0.049476  
divcurves_pca_shortest_path_distance 0  0.900000  0.046398  
divcurves_shortest_path_distance     1  0.904167  0.049476  
divcurves_pca_shortest_path_distance 1  0.900000  0.046398  
divcurves_shortest_path_dist

# Results

In [2]:
res_dir = 'simulations/results/ablations/distributions/nreps'

In [3]:
res = pd.read_csv(f'{res_dir}/res_df.csv', index_col=0, header=0)

In [4]:
res

,silh_mean,db_mean,ch_mean,silh_std,db_std,ch_std,ari_agglo_mean,ari_agglo_std,ari_kmeans_mean,ari_kmeans_std,acc_mean,acc_std
divcurves_shortest_path_distance_nreps_1,0.421604,0.754183,192.720745,0.033456,0.058361,33.387778,0.315380,0.115946,0.500475,0.081177,0.858333,0.046398
divcurves_pca_shortest_path_distance_nreps_1,0.434270,0.732491,197.008837,0.034245,0.058968,34.709484,0.322917,0.122553,0.502109,0.078131,0.870833,0.054167
divcurves_shortest_path_distance_nreps_3,0.422924,0.750646,193.447967,0.033879,0.059376,33.239422,0.310446,0.110164,0.502467,0.078777,0.904167,0.041874
divcurves_pca_shortest_path_distance_nreps_3,0.432234,0.734302,196.322620,0.034513,0.059587,34.101707,0.314835,0.114094,0.503326,0.079493,0.891667,0.046398
divcurves_shortest_path_distance_nreps_5,0.425761,0.747883,195.138790,0.033980,0.059903,34.090992,0.313527,0.108197,0.496749,0.077670,0.904167,0.049476
divcurves_pca_shortest_path_distance_nreps_5,0.434439,0.732061,197.968147,0.034544,0.060014,34.963096,0.329558,0.121440,0.498456,0.074992,0.900000,0.046398


In [5]:
res['nrep'] = res.index.map(lambda x: int(x[-1]))

In [6]:
res_multicol = pd.MultiIndex.from_product(
    [res.columns, res.index.map(lambda x: int(x[-1]))],
    names = ['score', 'nrep']
    )

## after implementation different seeds

In [52]:
res_dir = 'simulations/results/ablations/distributions/nreps/test'
res_unordered = pd.read_csv(f'{res_dir}/res_df.csv', index_col=[0,1], header=[0,1])


In [53]:
res_unordered

,,silh_mean,db_mean,ch_mean,silh_std,db_std,ch_std,ari_agglo_mean,ari_agglo_std,ari_kmeans_mean,ari_kmeans_std,...,ch_mean,silh_std,db_std,ch_std,ari_agglo_mean,ari_agglo_std,ari_kmeans_mean,ari_kmeans_std,acc_mean,acc_std
,,1,1,1,1,1,1,1,1,1,1,...,5,5,5,5,5,5,5,5,5,5
divcurves_shortest_path_distance,0,0.421604,0.754183,192.720745,0.033456,0.058361,33.387778,0.315380,0.115946,0.500475,0.081177,...,195.138790,0.033980,0.059903,34.090992,0.313527,0.108197,0.496749,0.077670,0.904167,0.049476
divcurves_pca_shortest_path_distance,0,0.434270,0.732491,197.008837,0.034245,0.058968,34.709484,0.322917,0.122553,0.502109,0.078131,...,197.968147,0.034544,0.060014,34.963096,0.329558,0.121440,0.498456,0.074992,0.900000,0.046398
divcurves_shortest_path_distance,1,0.418545,0.762476,190.735427,0.033549,0.059957,33.126113,0.315504,0.105647,0.498360,0.080853,...,195.138790,0.033980,0.059903,34.090992,0.313527,0.108197,0.496749,0.077670,0.904167,0.049476
divcurves_pca_shortest_path_distance,1,0.430249,0.742956,194.602720,0.034384,0.060510,34.370930,0.314268,0.115486,0.498357,0.080505,...,197.968147,0.034544,0.060014,34.963096,0.329558,0.121440,0.498456,0.074992,0.900000,0.046398
divcurves_shortest_path_distance,2,0.418545,0.762476,190.735427,0.033549,0.059957,33.126113,0.315504,0.105647,0.498360,0.080853,...,195.138790,0.033980,0.059903,34.090992,0.313527,0.108197,0.496749,0.077670,0.904167,0.049476
divcurves_pca_shortest_path_distance,2,0.430249,0.742956,194.602720,0.034384,0.060510,34.370930,0.314268,0.115486,0.498357,0.080505,...,197.968147,0.034544,0.060014,34.963096,0.329558,0.121440,0.498456,0.074992,0.900000,0.046398


In [57]:
#reorder
scores = list(res_unordered.columns.levels[0])
n_reps = list(res_unordered.columns.levels[1])
n_reps
res = res_unordered.loc[:,pd.MultiIndex.from_product([scores,n_reps])]

In [58]:
res

acc_mean                      \
                                               1         3         5   
divcurves_shortest_path_distance     0  0.858333  0.904167  0.904167   
divcurves_pca_shortest_path_distance 0  0.870833  0.891667  0.900000   
divcurves_shortest_path_distance     1  0.891667  0.904167  0.904167   
divcurves_pca_shortest_path_distance 1  0.887500  0.891667  0.900000   
divcurves_shortest_path_distance     2  0.891667  0.904167  0.904167   
divcurves_pca_shortest_path_distance 2  0.887500  0.891667  0.900000   

                                         acc_std                      \
                                               1         3         5   
divcurves_shortest_path_distance     0  0.046398  0.041874  0.049476   
divcurves_pca_shortest_path_distance 0  0.054167  0.046398  0.046398   
divcurves_shortest_path_distance     1  0.050000  0.041874  0.049476   
divcurves_pca_shortest_path_distance 1  0.049476  0.046398  0.046398   
divcurves_shortest_path_distance     2  0.050000  0.041874  0.049476   
divcurves_pca_shortest_path_distance 2  0.049476  0.046398  0.046398   

                                       ari_agglo_mean                      \
                                                    1         3         5   
divcurves_shortest_path_distance     0       0.315380  0.310446  0.313527   
divcurves_pca_shortest_path_distance 0       0.322917  0.314835  0.329558   
divcurves_shortest_path_distance     1       0.315504  0.310446  0.313527   
divcurves_pca_shortest_path_distance 1       0.314268  0.314835  0.329558   
divcurves_shortest_path_distance     2       0.315504  0.310446  0.313527   
divcurves_pca_shortest_path_distance 2       0.314268  0.314835  0.329558   

                                       ari_agglo_std  ...   db_mean    db_std  \
                                                   1  ...         5         1   
divcurves_shortest_path_distance     0      0.115946  ...  0.747883  0.058361   
divcurves_pca_shortest_path_distance 0      0.122553  ...  0.732061  0.058968   
divcurves_shortest_path_distance     1      0.105647  ...  0.747883  0.059957   
divcurves_pca_shortest_path_distance 1      0.115486  ...  0.732061  0.060510   
divcurves_shortest_path_distance     2      0.105647  ...  0.747883  0.059957   
divcurves_pca_shortest_path_distance 2      0.115486  ...  0.732061  0.060510   

                                                           silh_mean  \
                                               3         5         1   
divcurves_shortest_path_distance     0  0.059376  0.059903  0.421604   
divcurves_pca_shortest_path_distance 0  0.059587  0.060014  0.434270   
divcurves_shortest_path_distance     1  0.059376  0.059903  0.418545   
divcurves_pca_shortest_path_distance 1  0.059587  0.060014  0.430249   
divcurves_shortest_path_distance     2  0.059376  0.059903  0.418545   
divcurves_pca_shortest_path_distance 2  0.059587  0.060014  0.430249   

                                                            silh_std  \
                                               3         5         1   
divcurves_shortest_path_distance     0  0.422924  0.425761  0.033456   
divcurves_pca_shortest_path_distance 0  0.432234  0.434439  0.034245   
divcurves_shortest_path_distance     1  0.422924  0.425761  0.033549   
divcurves_pca_shortest_path_distance 1  0.432234  0.434439  0.034384   
divcurves_shortest_path_distance     2  0.422924  0.425761  0.033549   
divcurves_pca_shortest_path_distance 2  0.432234  0.434439  0.034384   

                                                            
                                               3         5  
divcurves_shortest_path_distance     0  0.033879  0.033980  
divcurves_pca_shortest_path_distance 0  0.034513  0.034544  
divcurves_shortest_path_distance     1  0.033879  0.033980  
divcurves_pca_shortest_path_distance 1  0.034513  0.034544  
divcurves_shortest_path_distance     2  0.033879  0.033980  
divcurves_pca_shortest_path_distance 2 